# Evidence-Attributed Dense Rubric RL

Train **Qwen3.5-4B** on **HealthBench** with a rubric judge (**Gemini 3.5 Flash**) that returns not just a score, but **verbatim evidence quotes with character offsets**. Those quotes become **per-token loss weights**, so the gradient concentrates on the exact sentences that earned (or lost) the points.

**The core rules:**

1. The scalar weighted rubric score is the **only** source of advantages (reward stays untouched).
2. Evidence spans only **redistribute** the per-token loss — mean token weight is renormalized to exactly 1.0.
3. Positive advantage → boost tokens proving a **positive criterion was MET** (push harder on the good parts).
4. Negative advantage → boost tokens proving a **negative criterion was MET** (suppress harder on the errors).
5. Broad criteria (tone, completeness) still score, but contribute no spans.
6. Every span is validated against the completion: exact offsets → `find()` fallback → dropped (score kept).

**Algorithm:** CISPO loss with clip-higher (MiniMax-M1) + DAPO soft overlong punishment (no dynamic sampling) + GDPO-style decoupled advantage normalization for the two reward streams (rubric quality, length penalty). No KL term, no reference model.

**Runtime:** one GPU (Colab A100 40GB target). No vLLM, no DeepSpeed — generation and training run sequentially on the same model. Slower, but no CUDA-version conflicts.


## 0 · Environment check

We deliberately do **not** install vLLM or DeepSpeed: both pin specific torch/CUDA builds and regularly break Colab. Everything below is plain `transformers` + `torch`. Sections 0–7 run on CPU (your laptop); sections 8+ need the GPU.


In [ ]:
!nvidia-smi || echo "no NVIDIA GPU — CPU smoke-test mode (sections 0-7 only)"


In [ ]:
import torch, transformers

ON_GPU = torch.cuda.is_available()
print(f"torch {torch.__version__} | cuda: {torch.version.cuda} | gpu available: {ON_GPU}")
print(f"transformers {transformers.__version__}")
if ON_GPU:
    print(f"device: {torch.cuda.get_device_name(0)}")


## 1 · Install & keys

On a **fresh Colab box**: clone your repo first, `cd` into it, then run the installs. Locally (with the project venv already set up) you can skip the pip cell.

API keys: in Colab, add `GEMINI_API_KEY` (and optionally `WANDB_API_KEY`) in the **Secrets** sidebar; locally put them in `.env`.


In [ ]:
# Fresh Colab box only — uncomment:
# !git clone https://github.com/YOUR_USERNAME/dense-rubric.git
# %cd dense-rubric

# %pip install -q -r requirements.txt
# %pip install -q -e ./rubric

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # local: reads .env
try:
    from google.colab import userdata  # Colab: reads the Secrets sidebar
    for key in ("GEMINI_API_KEY", "WANDB_API_KEY"):
        if not os.getenv(key):
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                pass
except ImportError:
    pass

print("GEMINI_API_KEY set:", bool(os.getenv("GEMINI_API_KEY")))
print("WANDB_API_KEY set: ", bool(os.getenv("WANDB_API_KEY")))


## 2 · Config

Every knob in one place. The defaults target an A100 40GB; on a smaller GPU, drop `MODEL_NAME` to `Qwen/Qwen3.5-2B` and/or shrink `GEN_BATCH_SIZE` / `MICROBATCH_SIZE`.


In [ ]:
from pathlib import Path

# ----- model -----
MODEL_NAME = "Qwen/Qwen3.5-4B"

# ----- rollout -----
NUM_PROMPTS_PER_ITERATION = 8     # unique HealthBench questions per iteration
GROUP_SIZE = 8                    # G: completions per question (the GRPO-style "group")
EPISODES_PER_ITERATION = NUM_PROMPTS_PER_ITERATION * GROUP_SIZE  # 64
MAX_RESPONSE_TOKENS = 1024
TEMPERATURE = 1.0
TOP_P = 1.0
GEN_BATCH_SIZE = 16               # sequences generated at once (bounds KV-cache memory)

# ----- evidence-span token weights -----
ALPHA = 0.5                       # max pre-clip boost if one criterion owned ALL rubric mass
MAX_BOOST = 1.0                   # cap on stacked boosts for a single token

# ----- advantages / anti-reward-hacking -----
LAMBDA_LEN = 0.3                  # weight of the overlong-penalty stream
OVERLONG_BUFFER = 256             # last N tokens before MAX_RESPONSE_TOKENS: penalty ramps 0 -> -1
DECOUPLED_NORM = True             # True = GDPO decoupled normalization; False = DAPO sum-then-norm
EPS_LOW, EPS_HIGH = 0.2, 0.4      # CISPO ratio clip (clip-HIGHER: eps_high > eps_low)

# ----- optimization -----
NUM_ITERATIONS = 1000
LEARNING_RATE = 1e-6
EPOCHS_PER_ITERATION = 2          # passes over each rollout batch; pass 2 is off-policy -> clip engages
MICROBATCH_SIZE = 2
MAX_GRAD_NORM = 1.0

# ----- data -----
MAX_PROMPT_TOKENS = 1024
TRAIN_SPLIT_FRAC = 0.9
SEED = 42

# ----- eval / logging / checkpoints -----
EVAL_EVERY = 10                   # HealthBench eval on the validation split every N iterations
NUM_EVAL_SAMPLES = 64
JUDGE_MAX_CONCURRENT = 8          # completions graded at once (x ~10 criteria each ~= 80 calls in flight)
CHECKPOINT_EVERY = 25             # Colab can die at any time
EXP_DIR = Path("experiments/dense-rubric")
# Survive Colab disconnects — keep checkpoints on Drive instead:
# from google.colab import drive; drive.mount("/content/drive")
# EXP_DIR = Path("/content/drive/MyDrive/dense-rubric")


## 3 · Dataset — HealthBench (5,000 examples, 90/10 split)

Each example is a patient conversation plus a physician-written rubric: a list of criteria, each worth positive points (desired behavior) or negative points (errors). The official score is `earned points / total positive points`, clamped to [0, 1].


In [ ]:
from transformers import AutoTokenizer

import utils

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

train_raw, val_raw = utils.load_healthbench_dataset(
    cache_dir="data", split_frac=TRAIN_SPLIT_FRAC, seed=SEED
)
train_examples = utils.prepare_examples(train_raw, tokenizer, MAX_PROMPT_TOKENS)
val_examples = utils.prepare_examples(val_raw, tokenizer, MAX_PROMPT_TOKENS)
print(f"after prompt-length filter: {len(train_examples)} train / {len(val_examples)} val")


In [ ]:
# peek at one example + how many criteria rubrics typically have
import numpy as np

ex = train_examples[0]
print("PROMPT (last user turn):", ex["messages"][-1]["content"][:400], "...")
print()
for r in ex["rubrics"][:5]:
    print(f"  [{r['points']:+d} pts] {r['criterion'][:110]}")
print(f"  ... ({len(ex['rubrics'])} criteria total)")

counts = [len(e["rubrics"]) for e in train_examples]
print(f"criteria per example: mean {np.mean(counts):.1f}, median {np.median(counts):.0f}, "
      f"min {min(counts)}, max {max(counts)}")


## 4 · Judge setup

Two graders built from the (locally cloned + extended) `rubric` package:

- **`train_grader`** — `SpanPerCriterionGrader`: one Gemini call per criterion, returns MET/UNMET **plus evidence spans**, which are validated against the completion immediately.
- **`eval_grader`** — plain `PerCriterionGrader`: scalar only. Eval produces no gradients, so spans would just add cost and failure modes; this keeps `eval/healthbench_score` identical to official HealthBench grading.


In [ ]:
from rubric.autograders import PerCriterionGrader, SpanPerCriterionGrader

from utils import gemini_scalar_generate_fn, gemini_span_generate_fn

train_grader = SpanPerCriterionGrader(generate_fn=gemini_span_generate_fn, normalize=True)
eval_grader = PerCriterionGrader(generate_fn=gemini_scalar_generate_fn, normalize=True)
print("graders ready (judge model:", os.getenv("DR_JUDGE_MODEL", utils.DEFAULT_JUDGE_MODEL), ")")


## 5 · Smoke tests (CPU-only)

Quick assertions that the whole span → token → weight pipeline is correct, **before** spending GPU time or Gemini calls. All of these run on a laptop.

### 5a · Span validation

The judge promises `completion[start:end] == quote`, but LLMs get offsets wrong constantly. Validation ladder: exact offset → `find()` (first occurrence) → whitespace-normalized find → drop the span (the criterion's score is kept either way).


In [ ]:
from rubric import EvidenceSpan, validate_spans

text = ("I understand this is scary. Call 911 or go to the ER right away. "
        "Chest pain can be serious. Do not drive yourself.")

q1 = "Call 911 or go to the ER right away."
s1 = text.find(q1)
exact = EvidenceSpan(quote=q1, start_char=s1, end_char=s1 + len(q1))           # correct offsets
wrong = EvidenceSpan(quote="Chest pain can be serious.", start_char=0, end_char=26)  # wrong offsets
fake = EvidenceSpan(quote="Take two aspirin and rest tonight.", start_char=0, end_char=35)  # hallucinated

valid, stats = validate_spans(text, [exact, wrong, fake])
print("validated spans:", [(s, e, text[s:e]) for s, e in valid])
print("stats:", stats)

assert stats["n_exact_offset"] == 1
assert stats["n_recovered_by_find"] == 1   # wrong offsets, but the quote IS in the text
assert stats["n_dropped"] == 1             # hallucinated quote -> dropped, score unaffected
assert all(text[s:e] for s, e in valid)
print("span validation OK")


### 5b · Characters → tokens

Spans live in _character_ space; the loss lives in _token_ space. `token_char_ranges` maps each sampled token id to the character range it occupies in the decoded string — via incremental prefix decoding, which is guaranteed consistent with the exact string the judge graded (re-encoding the text could produce _different_ tokens than the ones actually sampled).


In [ ]:
from utils import token_char_ranges

sample = "The naïve approach: take 100 mg/day of CoQ10 — it's naïve, but common."  # unicode + dosage
ids = tokenizer(sample, add_special_tokens=False)["input_ids"]
decoded = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
ranges = token_char_ranges(ids, tokenizer)

assert len(ranges) == len(ids)
assert ranges[-1][1] == len(decoded)                                    # covers the full string
assert all(a[1] == b[0] for a, b in zip(ranges, ranges[1:]))            # contiguous, no gaps

# substring round-trip: the tokens overlapping a char span must contain that substring
sub = "100 mg/day"
s = decoded.find(sub)
covering = [i for i, (cs, ce) in enumerate(ranges) if cs < s + len(sub) and ce > s]
assert sub in tokenizer.decode([ids[i] for i in covering])

for i in covering:
    cs, ce = ranges[i]
    print(f"token {i:3d}  chars [{cs:3d},{ce:3d})  {decoded[cs:ce]!r}")
print("char->token mapping OK")


### 5c · Evidence spans → token weights

`build_token_weights` implements the credit-redistribution contract: base 1.0, boost `alpha_c = ALPHA * |w_c| / Σ|w|` on covered tokens (clipped at `MAX_BOOST`), then renormalize so the **mean is exactly 1.0** — the gradient scale is identical to vanilla, only the distribution changes. Sign gating: positive advantage boosts MET-positive spans, negative advantage boosts MET-negative (failure) spans.


In [ ]:
from rubric import EvaluationReport, SpanCriterionReport

from utils import build_token_weights

completion = ("Drink plenty of fluids and rest. See a doctor within 24 hours if the fever "
              "persists or gets worse. Avoid strenuous exercise this week.")
comp_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
comp_ranges = token_char_ranges(comp_ids, tokenizer)

quote = "See a doctor within 24 hours"
qs = completion.find(quote)
fake_report = EvaluationReport(score=0.8, report=[
    SpanCriterionReport(weight=5.0, requirement="Advises seeing a doctor", verdict="MET",
                        reason="", valid_spans=[(qs, qs + len(quote))]),
    SpanCriterionReport(weight=2.0, requirement="Recommends hydration", verdict="MET",
                        reason="", valid_spans=[]),          # MET but no surviving span
    SpanCriterionReport(weight=-3.0, requirement="Recommends antibiotics", verdict="UNMET",
                        reason=""),
])

w_pos, st = build_token_weights(len(comp_ids), comp_ranges, fake_report,
                                advantage=+1.0, alpha=ALPHA, max_boost=MAX_BOOST)
assert abs(sum(w_pos) / len(w_pos) - 1.0) < 1e-9, "mean weight must be exactly 1.0"
boosted = [i for i, (cs, ce) in enumerate(comp_ranges) if cs < qs + len(quote) and ce > qs]
assert all(w_pos[i] > w_pos[0] for i in boosted), "span tokens must outweigh non-span tokens"
print(f"positive-advantage weights: boosted {st['fraction_tokens_boosted']:.0%} of tokens, "
      f"max weight {st['max_token_weight']:.3f}")

# negative advantage: the MET-positive span no longer applies -> all ones (vanilla)
w_neg, _ = build_token_weights(len(comp_ids), comp_ranges, fake_report,
                               advantage=-1.0, alpha=ALPHA, max_boost=MAX_BOOST)
assert w_neg == [1.0] * len(comp_ids)

# judge failure / no report -> all ones (degrades to vanilla GRPO-style loss)
w_none, _ = build_token_weights(len(comp_ids), comp_ranges, None, advantage=1.0)
assert w_none == [1.0] * len(comp_ids)
print("token-weight construction OK")


### 5d · Overlong penalty + decoupled (GDPO) vs summed (GRPO/DAPO) advantages

The demo below shows **why** decoupled normalization matters: with summed rewards, the high-variance length penalty _drowns_ the small-but-real quality differences. Decoupled normalization scores each stream against its own spread first, preserving quality resolution.


In [ ]:
from utils import compute_advantages, overlong_penalty

# DAPO soft overlong punishment: 0 until the buffer, linear ramp to -1, -1 if truncated
assert overlong_penalty(100, False, 1024, 256) == 0.0
assert overlong_penalty(896, False, 1024, 256) == -0.5   # halfway into the buffer
assert overlong_penalty(1024, False, 1024, 256) == -1.0
assert overlong_penalty(50, True, 1024, 256) == -1.0     # truncated -> full penalty

# one group of 4 completions: quality differences are small (0.9 vs 0.8) but REAL;
# length penalties are large (0 vs -1)
scores = [0.9, 0.8, 0.9, 0.8]
pens = [0.0, -1.0, -1.0, 0.0]

adv_sum = compute_advantages(scores, pens, group_size=4, lambda_len=0.3, decoupled=False)
adv_dec = compute_advantages(scores, pens, group_size=4, lambda_len=0.3, decoupled=True)
print("completion:        (0.9, ok)  (0.8, long)  (0.9, long)  (0.8, ok)")
print("summed (GRPO):   ", np.round(adv_sum, 2))
print("decoupled (GDPO):", np.round(adv_dec, 2))

# summed: the long high-quality completion (idx 2) ranks BELOW the short low-quality one
# (idx 3) — the length penalty drowned the quality signal
assert adv_sum[2] < adv_sum[3]
# decoupled: quality survives — idx 2 still outranks idx 3
assert adv_dec[2] > adv_dec[3]
# both modes: zero-variance group -> zero advantages, never NaN
assert np.allclose(compute_advantages([0.5] * 4, [0.0] * 4, 4), 0.0)
print("advantage computation OK")


### 5e · Live judge call (optional, needs `GEMINI_API_KEY`)

One real grading call on a real HealthBench example with a hand-written answer. Prints every criterion verdict and the completion with validated evidence spans highlighted as `«...»`.


In [ ]:
from utils import make_criteria

if os.getenv("GEMINI_API_KEY"):
    live_ex = val_examples[0]
    live_completion = (
        "I'm sorry you're dealing with this. Based on what you describe, you should "
        "see a doctor within the next day or two to get it checked properly. In the "
        "meantime, rest, stay hydrated, and avoid taking any leftover antibiotics. "
        "If you develop severe pain, a high fever, or trouble breathing, go to the "
        "emergency department right away."
    )
    live_report = await train_grader.grade(
        live_completion, make_criteria(live_ex["rubrics"]), query=live_ex["query"]
    )
    print(f"score: {live_report.score:.3f}  (raw weighted sum: {live_report.raw_score})\n")
    spans = []
    for cr in live_report.report:
        print(f"[{cr.weight:+.0f}] {cr.verdict:5s} loc={str(cr.localizable):5s} "
              f"spans={len(cr.valid_spans)} | {cr.requirement[:90]}")
        spans += cr.valid_spans

    highlighted = live_completion
    for s, e in sorted(set(spans), reverse=True):   # right-to-left so offsets stay valid
        highlighted = highlighted[:e] + "»" + highlighted[e:]
        highlighted = highlighted[:s] + "«" + highlighted[s:]
    print("\n" + highlighted)
else:
    print("GEMINI_API_KEY not set — skipping live judge call")


## 6 · Walkthrough: how credit gets redistributed

Token-by-token view of the 5c example. Tokens inside the evidence span carry weight > 1; everything else drops slightly below 1 (the renormalization). Total gradient mass is unchanged.


In [ ]:
print(f"{'token':>6} | {'weight':>7} | text")
print("-" * 50)
for i, ((cs, ce), w) in enumerate(zip(comp_ranges, w_pos)):
    marker = " <-- evidence" if w > 1.0 else ""
    print(f"{i:>6} | {w:>7.3f} | {completion[cs:ce]!r}{marker}")
print(f"\nmean weight = {sum(w_pos)/len(w_pos):.6f} (always exactly 1.0)")


## 7 · The loss — CISPO with clip-higher + evidence weights

$$\mathcal{L} = -\frac{1}{\sum_t m_t}\sum_t \; \underbrace{\mathrm{clip}\big(r_t,\,1-\varepsilon_{low},\,1+\varepsilon_{high}\big)\,\hat A}_{\text{detached coefficient}} \; \cdot\; w_t \,\log \pi_\theta(y_t)\,, \qquad r_t = \frac{\pi_\theta(y_t)}{\pi_{old}(y_t)}$$

- **CISPO vs PPO clipping:** PPO clips the _update_, so a clipped token contributes **zero** gradient — and the clipped tokens are often exactly the rare, high-leverage ones. CISPO instead clips the _ratio inside a detached coefficient_: every token always keeps a gradient (`-coef · ∇log π`); clipping only caps how strongly off-policy tokens count.
- **Clip-higher** (`ε_high > ε_low`, from DAPO): lets low-probability tokens grow further before capping → fights entropy collapse.
- **`w_t`** is the evidence weight from section 5c (mean 1.0 per completion) — credit redistribution only.
- **No KL term, no reference model** (CISPO and DAPO both drop it) — saves ~8 GB.
- `old_logp` is computed once per iteration right after generation; with `EPOCHS_PER_ITERATION = 2`, the second pass is genuinely off-policy, which is when the clip matters.

Implementation: `utils.compute_cispo_loss`.


## 8 · Model (GPU from here on)

Single bf16 copy of the policy + gradient checkpointing + 8-bit paged AdamW. Rough A100-40GB budget: weights 8 GB + grads 8 GB + 8-bit optimizer state ~8 GB + activations/KV ≈ comfortably under 40 GB. On L4/T4: use `Qwen/Qwen3.5-2B`.


In [ ]:
from transformers import AutoModelForCausalLM

assert ON_GPU, "sections 8+ need a CUDA GPU"
device = torch.device("cuda")

policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, attn_implementation="sdpa"
).to(device)
policy_model.gradient_checkpointing_enable()  # trade compute for activation memory
policy_model.config.use_cache = False         # checkpointing-incompatible; generate() re-enables it

try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.PagedAdamW8bit(policy_model.parameters(), lr=LEARNING_RATE)
    print("optimizer: PagedAdamW8bit (8-bit state, pages to CPU under pressure)")
except ImportError:
    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=LEARNING_RATE)
    print("WARNING: bitsandbytes unavailable -> fp32 AdamW needs ~32GB of optimizer state")

print(f"params: {sum(p.numel() for p in policy_model.parameters()) / 1e9:.2f}B")


## 9 · Checkpoint resume

Plain `save_pretrained` + optimizer state. If a checkpoint exists in `EXP_DIR`, resume from it (Colab kills sessions without warning).


In [ ]:
from utils import find_last_checkpoint, load_checkpoint

start_iteration = 0
ckpt_path, ckpt_iter = find_last_checkpoint(EXP_DIR)
if ckpt_path is not None:
    start_iteration = load_checkpoint(ckpt_path, policy_model, optimizer)
    print(f"resumed from {ckpt_path} at iteration {start_iteration}")
else:
    print("no checkpoint found — starting fresh")


## 10 · Training loop

Per iteration: _(eval every 10)_ → sample 8 prompts → generate 8 completions each → grade with the span judge → advantages (GDPO-decoupled: rubric + overlong streams) → evidence token weights → 2 CISPO epochs over microbatches → log everything (including the full criteria + spans tables) → checkpoint every 25.

The Gemini grading (~640 calls) runs with top-level `await` — Jupyter/Colab support it natively.


In [ ]:
import random
import time

import wandb

from utils import (
    calculate_advantages_and_weights,
    compute_cispo_loss,
    compute_token_log_probs,
    evaluate_on_validation,
    generate_completions,
    grade_completions,
    log_criteria_and_spans_table,
    make_criteria,
    prepare_model_inputs,
    save_checkpoint,
)

run = wandb.init(
    project="dense-rubric",
    config={k: v for k, v in globals().items()
            if k.isupper() and isinstance(v, (int, float, str, bool))},
    resume="allow",
)


def iter_microbatches(model_inputs, batch_size, order=None):
    n = model_inputs["input_ids"].shape[0]
    idx = list(range(n)) if order is None else list(order)
    for start in range(0, n, batch_size):
        rows = idx[start : start + batch_size]
        yield {k: v[rows] for k, v in model_inputs.items()}


rng = random.Random(SEED + start_iteration)


In [ ]:
for iteration in range(start_iteration, NUM_ITERATIONS):
    t_iter = time.time()

    ############################################################
    # EVAL — official HealthBench score on the val split
    ############################################################
    if iteration % EVAL_EVERY == 0:
        eval_metrics, eval_rows = await evaluate_on_validation(
            policy_model, tokenizer, val_examples, eval_grader,
            num_eval_samples=NUM_EVAL_SAMPLES, max_new_tokens=MAX_RESPONSE_TOKENS,
            gen_batch_size=GEN_BATCH_SIZE, max_concurrent_completions=JUDGE_MAX_CONCURRENT,
        )
        eval_table = wandb.Table(columns=["prompt_id", "completion", "score"])
        for row in eval_rows:
            eval_table.add_data(row["prompt_id"], row["completion"], row["score"])
        run.log({**eval_metrics, "tables/eval": eval_table}, step=iteration)
        print(f"[iter {iteration:4d}] EVAL healthbench_score = "
              f"{eval_metrics['eval/healthbench_score']:.3f}")

    ############################################################
    # ROLLOUT — G completions per prompt (no grad, chunked)
    ############################################################
    batch = rng.sample(train_examples, NUM_PROMPTS_PER_ITERATION)
    query_ids, response_ids, finish_reasons = generate_completions(
        policy_model, tokenizer, [ex["prompt_token_ids"] for ex in batch],
        group_size=GROUP_SIZE, max_new_tokens=MAX_RESPONSE_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, gen_batch_size=GEN_BATCH_SIZE,
    )
    torch.cuda.empty_cache()  # drop generate's KV cache before the training pass

    # decode ONCE with these exact flags: this string is what the judge grades AND
    # what every character offset refers to (must match token_char_ranges)
    completions = [
        tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
        for ids in response_ids
    ]

    ############################################################
    # GRADE — scalar rubric score + validated evidence spans
    ############################################################
    def flat(key):
        return [ex[key] for ex in batch for _ in range(GROUP_SIZE)]

    reports, judge_stats = await grade_completions(
        completions, [make_criteria(r) for r in flat("rubrics")], flat("query"),
        train_grader, max_concurrent_completions=JUDGE_MAX_CONCURRENT,
    )

    ############################################################
    # ADVANTAGES (scalar reward only!) + TOKEN WEIGHTS (spans only!)
    ############################################################
    rewards, advantages, token_weights, reward_stats = calculate_advantages_and_weights(
        response_ids, finish_reasons, reports, tokenizer,
        group_size=GROUP_SIZE, max_response_tokens=MAX_RESPONSE_TOKENS,
        overlong_buffer=OVERLONG_BUFFER, lambda_len=LAMBDA_LEN,
        alpha=ALPHA, max_boost=MAX_BOOST, decoupled_norm=DECOUPLED_NORM,
    )

    ############################################################
    # TRAIN — CISPO, EPOCHS_PER_ITERATION optimizer steps
    ############################################################
    model_inputs = prepare_model_inputs(
        query_ids, response_ids, advantages.tolist(), token_weights,
        device, tokenizer.pad_token_id,
    )
    total_response_tokens = model_inputs["labels_mask"][:, 1:].sum().item()

    with torch.no_grad():  # old-policy log-probs: the reference point for the CISPO ratio
        model_inputs["old_logps"] = torch.cat([
            compute_token_log_probs(policy_model, mb, TEMPERATURE)
            for mb in iter_microbatches(model_inputs, MICROBATCH_SIZE)
        ])

    policy_model.train()
    loss_metrics, grad_norms = [], []
    order = list(range(len(response_ids)))
    for epoch in range(EPOCHS_PER_ITERATION):
        rng.shuffle(order)
        optimizer.zero_grad(set_to_none=True)
        for mb in iter_microbatches(model_inputs, MICROBATCH_SIZE, order):
            loss, m = compute_cispo_loss(
                policy_model, mb, total_response_tokens,
                temperature=TEMPERATURE, eps_low=EPS_LOW, eps_high=EPS_HIGH,
            )
            loss.backward()   # grads accumulate; loss already normalized by full-batch tokens
            loss_metrics.append(m)
        grad_norms.append(
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), MAX_GRAD_NORM).item()
        )
        optimizer.step()

    ############################################################
    # LOG — scalars + full criteria/span tables, every iteration
    ############################################################
    run.log({
        **reward_stats, **judge_stats,
        "train/loss": float(np.mean([m["loss"] for m in loss_metrics])),
        "train/ratio_mean": float(np.mean([m["ratio_mean"] for m in loss_metrics])),
        "train/clip_fraction": float(np.mean([m["clip_fraction"] for m in loss_metrics])),
        "train/grad_norm": float(np.mean(grad_norms)),
        "train/iteration_seconds": time.time() - t_iter,
    }, step=iteration)
    log_criteria_and_spans_table(
        run, iteration, flat("prompt_id"), completions, reports, rewards, advantages
    )
    print(f"[iter {iteration:4d}] reward {reward_stats['train/reward']:.3f} | "
          f"span validity {reward_stats['span/validity_rate']:.0%} | "
          f"boosted {reward_stats['span/fraction_tokens_boosted']:.0%} of tokens | "
          f"{time.time() - t_iter:.0f}s")

    if (iteration + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(policy_model, optimizer, iteration + 1, EXP_DIR)


## 11 · Notes & future work

**Reading the span metrics:**

- `span/validity_rate` — share of judge spans that survived validation. If it sinks below ~60%, tighten the span instructions (fewer/shorter quotes). Low validity degrades _gracefully_: fewer boosts → closer to vanilla loss; rewards are never affected.
- `span/exact_offset_rate` — how often the judge's offsets were literally correct. Expect this to be low; `find()` is the workhorse.
- `tables/criteria_spans` — every criterion verdict + every surviving span, every iteration. **This is the reward-hacking watchtower**: keyword stuffing and rubric-parroting show up here as suspicious quotes long before the scalar reward looks wrong.

**Anti-reward-hacking measures in play:** soft overlong punishment (length hacking), clip-higher (entropy collapse), zero-variance guards (degenerate groups), truncation = full −1 penalty, full judge-output logging (human-auditable).

**Things to try next:**

- A/B `DECOUPLED_NORM` True vs False and compare `eval/healthbench_score` curves.
- Sweep `ALPHA` (0 = vanilla baseline — always run this control!).
- `EPOCHS_PER_ITERATION` 1 vs 2 vs 4 (more off-policy reuse per rollout = cheaper but riskier).
- Negative-span suppression on _positive_-advantage completions (currently sign-gated off by design).
